# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imalik-7/Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

My provisional lane is Refresh / Content Opportunity Scoring. I chose this lane because the useful problem is not simply to train a model, but to help a content team decide which pages deserve review first when time is limited. I want to use observable search, content, freshness, and engagement signals to rank pages that may need attention. The final result should support human decisions rather than automatically decide that a page must be changed.## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [4]:
import os
import sys
import subprocess
import pandas as pd

# Official starter repo
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Colab needs the whole repo, not only this notebook
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

assert os.path.exists(DATA_PATH), "Starter CSV not found."

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Working directory:", os.getcwd())
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Working directory: /content/flyrank-ml-internship-starter
Rows: 30000
Columns: 44


Which content pages should be reviewed first for refresh or monitoring based on observable search visibility, performance direction, age, freshness, CTR, position, and engagement signals?

The unit of analysis is one content page.

The decision I want to improve is which pages the content or SEO team should spend its limited review time on first.

The output will be a ranked list of review candidates, ideally with simple reason codes explaining why each page received its position.

A human reviewer could then decide whether to refresh, expand, protect, prune, or monitor a page.

A wrong recommendation has a real cost. It could waste time on a page that did not need attention, lead to unnecessary changes, or cause the team to miss another page with a stronger opportunity. Because of this, the system should prioritize review rather than claim that an action is guaranteed to work.

In [5]:
print("Total rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())

duplicate_pages = len(df) - df["content_id"].nunique()
print("Duplicate content_id rows:", duplicate_pages)

Total rows: 30000
Unique content pages: 30000
Duplicate content_id rows: 0


I want to check whether this lane has enough real examples to be useful. I will look at the number of eligible pages, how many are currently marked as declining in the starter data, and how many have at least 500 impressions in the last 90 days. These numbers help show whether there is a meaningful pool of pages that could be prioritized for review.

In [6]:
# Section 3: Quick look at the data

eligible = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

total_eligible = len(eligible)

declining_pages = (eligible["trend_direction"] == "down").sum()
declining_pct = declining_pages / total_eligible * 100

visible_pages = (eligible["impressions_90d"] >= 500).sum()
visible_pct = visible_pages / total_eligible * 100

print(f"Eligible pages: {total_eligible:,}")
print(f"Declining pages: {declining_pages:,} ({declining_pct:.1f}%)")
print(f"Pages with at least 500 impressions: {visible_pages:,} ({visible_pct:.1f}%)")

Eligible pages: 30,000
Declining pages: 16,262 (54.2%)
Pages with at least 500 impressions: 16,726 (55.8%)


This project can identify observed patterns and associations in the available search, content, freshness, and engagement data. It can help rank pages that may deserve human review before other pages.

The result should be treated as decision support, not proof that a page must be changed.

I cannot claim that refreshing a recommended page will cause its traffic, rankings, CTR, or engagement to improve. The dataset is observational, so it does not prove causation.

I also cannot claim that this project discovers or predicts Google's ranking algorithm. A highly ranked page in my output means that the available evidence suggests it may deserve review first, not that an action is guaranteed to succeed.

The guide makes the same distinction: the goal is normally to find the right page to review first, not to guarantee that editing a page will make it recover.

In [7]:
# Section 4: Careful words / public-safety check

sensitive_names = [
    "client_name",
    "domain",
    "raw_url",
    "raw_query",
    "raw_title"
]

found_sensitive_fields = [
    col for col in sensitive_names
    if col in df.columns
]

if found_sensitive_fields:
    print("Potential sensitive fields found:", found_sensitive_fields)
else:
    print("No obvious raw client/domain/URL/query/title fields found.")

No obvious raw client/domain/URL/query/title fields found.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.